In [ ]:
import pandas as pd
import numpy as np
import csv



In [ ]:
# Read CSV and convert to NumPy array
review_df = pd.read_csv('rotten_tomatoes_critic_reviews.csv')
review_df = review_df[['rotten_tomatoes_link','review_content']]

review_df = review_df.dropna()
review_df_condensed = review_df.groupby('rotten_tomatoes_link', as_index=False)['review_content'].agg(' '.join)

In [ ]:
print(review_df_condensed.head(5))
print('--------------------')
print(review_df_condensed.loc[0, 'review_content'])


In [ ]:
movie_df = pd.read_csv('rotten_tomatoes_movies.csv')
movie_df = movie_df[['rotten_tomatoes_link','movie_title']]
movie_df_unique = movie_df.drop_duplicates(subset="rotten_tomatoes_link")

movie_and_reviews_df = review_df_condensed.merge(movie_df_unique, on="rotten_tomatoes_link", how="inner")
movie_and_reviews_df = movie_and_reviews_df[["movie_title", "review_content"]]


In [ ]:
print(movie_and_reviews_df.head(10))

print('--------------------')
print(movie_and_reviews_df.loc[0, 'review_content'])

In [ ]:
oscar_df = pd.read_csv('ML_dataset.csv')
oscar_df = oscar_df[['title','best_picture_nom']]
oscar_df_unique = oscar_df.drop_duplicates(subset="title")


movie_and_reviews_df.rename(columns={'movie_title': 'title'}, inplace=True)

master_df = movie_and_reviews_df.merge(oscar_df_unique, on="title", how="inner")
master_df["review_content"] =  master_df["review_content"].str.lower()

In [ ]:
print(master_df.head(50))

In [ ]:
#format an np array of tuples (review, label)
master_df = master_df.sample(frac=1, random_state=42).reset_index(drop=True) # shuffle before embeddings
master_tuples = list(
    master_df[["best_picture_nom", "review_content"]]
    .itertuples(index=False, name=None)
)
print(master_tuples[:5])

In [ ]:
def load_feature_dictionary(file):
    """
    Creates a map of words to vectors using the file that has the glove
    embeddings.

    Parameters:
        file (str): File path to the glove embedding file.

    Returns:
        A dictionary indexed by words, returning the corresponding glove
        embedding np.ndarray.
    """
    glove_map = dict()
    with open(file, encoding='utf-8') as f:
        read_file = csv.reader(f, delimiter='\t')
        for row in read_file:
            word, embedding = row[0], row[1:]
            glove_map[word] = np.array(embedding, dtype=float)
    return glove_map

In [ ]:
feature_dict = load_feature_dictionary("glove_embeddings.txt")
glove_get = feature_dict.get
rows = []

for label, review in master_tuples:
    vec = np.zeros(300)
    count = 0

    for word in review.split():
        wvec = glove_get(word)
        if wvec is not None:
            vec += wvec
            count += 1

    if count > 0:
        vec /= count  # find the average glove embedding

    rows.append([label, *vec])
    
output = np.array(rows)

np.savetxt(
    "formatted_glove_reviews.tsv",
    output,
    delimiter="\t",
    fmt="%.6f"
)

In [ ]:
df = pd.read_csv("formatted_glove_reviews.tsv", sep="\t", header=None)
n = len(df)
train_end = int(0.8 * n)
val_end = int(0.9 * n)
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]
train_df.to_csv("train.tsv", sep="\t", index=False, header=False)
val_df.to_csv("val.tsv", sep="\t", index=False, header=False)
test_df.to_csv("test.tsv", sep="\t", index=False, header=False)